# 🎭 Advanced Ensemble Methods & Model Stacking

## Overview
This notebook implements sophisticated ensemble methods to combine multiple models for superior salary prediction performance. We'll explore various ensemble techniques and create a robust meta-learning framework.

### Ensemble Techniques:
- 🎯 **Voting Regressors**: Simple and weighted voting
- 📚 **Bagging Methods**: Bootstrap aggregating
- 🚀 **Boosting Algorithms**: AdaBoost, Gradient Boosting, XGBoost
- 🏗️ **Stacking**: Multi-level model combination
- 🔄 **Blending**: Holdout-based ensemble
- 🎨 **Custom Ensembles**: Domain-specific combinations

### Advanced Strategies:
- 🎛️ **Dynamic Weighting**: Adaptive model weights
- 📊 **Cross-Validation Stacking**: Robust meta-learning
- 🔍 **Diversity Analysis**: Model complementarity assessment
- ⚡ **Performance Optimization**: Speed vs accuracy trade-offs

---

In [ ]:
# Import comprehensive ensemble learning libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn ensemble methods
from sklearn.ensemble import (
    VotingRegressor, BaggingRegressor, AdaBoostRegressor,
    RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor,
    StackingRegressor
)

# Base models for ensemble
from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, ElasticNet, 
    BayesianRidge, HuberRegressor
)
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.gaussian_process import GaussianProcessRegressor

# Advanced ensemble libraries
try:
    import xgboost as xgb
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
    print("✅ XGBoost available for advanced boosting")
except ImportError:
    XGBOOST_AVAILABLE = False
    print("⚠️ XGBoost not available. Install with: pip install xgboost")

try:
    import lightgbm as lgb
    from lightgbm import LGBMRegressor
    LIGHTGBM_AVAILABLE = True
    print("✅ LightGBM available for gradient boosting")
except ImportError:
    LIGHTGBM_AVAILABLE = False
    print("⚠️ LightGBM not available. Install with: pip install lightgbm")

try:
    import catboost as cb
    from catboost import CatBoostRegressor
    CATBOOST_AVAILABLE = True
    print("✅ CatBoost available for categorical boosting")
except ImportError:
    CATBOOST_AVAILABLE = False
    print("⚠️ CatBoost not available. Install with: pip install catboost")

# Model selection and evaluation
from sklearn.model_selection import (
    train_test_split, cross_val_score, KFold, 
    GridSearchCV, RandomizedSearchCV
)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, 
    r2_score, explained_variance_score
)

# Visualization and utilities
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML
import time
import joblib
from itertools import combinations

# Set random seed for reproducibility
np.random.seed(42)

print("\n🎭 Ensemble Methods Setup Complete!")
print(f"🚀 XGBoost: {XGBOOST_AVAILABLE}")
print(f"⚡ LightGBM: {LIGHTGBM_AVAILABLE}")
print(f"🐱 CatBoost: {CATBOOST_AVAILABLE}")
print("🎯 Ready for Advanced Ensemble Learning!")

In [ ]:
# Load and prepare data for ensemble methods
print("📥 PREPARING DATA FOR ENSEMBLE METHODS")
print("=" * 50)

# Load dataset
df = pd.read_csv('../ethiopia_salary_data.csv')
print(f"✅ Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns")

# Handle missing values
df['test_score'].fillna(df['test_score'].median(), inplace=True)

# Prepare features and target
feature_columns = ['experience_years', 'test_score', 'department', 'education_level', 'location']
target_column = 'salary_etb'

X = df[feature_columns].copy()
y = df[target_column].copy()

# Split data with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, 
    stratify=pd.cut(y, bins=5, labels=False)
)

# Further split training data for stacking (train/validation)
X_train_stack, X_val_stack, y_train_stack, y_val_stack = train_test_split(
    X_train, y_train, test_size=0.25, random_state=42
)

print(f"📊 Training samples (stacking): {X_train_stack.shape[0]}")
print(f"📊 Validation samples (stacking): {X_val_stack.shape[0]}")
print(f"📊 Testing samples: {X_test.shape[0]}")

# Create preprocessing pipeline
numerical_features = ['experience_years', 'test_score']
categorical_features = ['department', 'education_level', 'location']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
    ]
)

print(f"✅ Data preparation complete")
print(f"🎯 Target range: {y.min():,.0f} - {y.max():,.0f} ETB")

# Define evaluation function for ensemble methods
def evaluate_ensemble(model, X_train, X_test, y_train, y_test, model_name, cv_folds=5):
    """Comprehensive ensemble model evaluation"""
    start_time = time.time()
    
    # Fit model
    model.fit(X_train, y_train)
    
    # Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Cross-validation score
    cv_scores = cross_val_score(model, X_train, y_train, 
                               cv=cv_folds, scoring='neg_mean_squared_error')
    cv_rmse = np.sqrt(-cv_scores.mean())
    cv_rmse_std = np.sqrt(cv_scores.std())
    
    # Calculate metrics
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    
    training_time = time.time() - start_time
    
    return {
        'model_name': model_name,
        'train_rmse': train_rmse,
        'test_rmse': test_rmse,
        'cv_rmse': cv_rmse,
        'cv_rmse_std': cv_rmse_std,
        'train_r2': train_r2,
        'test_r2': test_r2,
        'train_mae': train_mae,
        'test_mae': test_mae,
        'training_time': training_time,
        'overfitting': train_r2 - test_r2,
        'generalization_gap': test_rmse - train_rmse
    }

print("\n🔧 Evaluation function defined")
print("🎭 Ready to build ensemble models!")